# CA1 Project 1: T14 Multi-Stop Route Planner [Travel]
**Agentic AI Course Project**

This notebook demonstrates an autonomous Multi-Stop Route Planner agent built with **LangChain**, **ChatGroq** (`qwen/qwen3.6-27b`), and **LangGraph MemorySaver**.

### Key Features:
- **Plan-Act Loop**: The agent inspects pairwise distances, evaluates route permutations, and commits the optimal visiting order using custom tools.
- **Tools Integrated**:
  1. `get_distance(a, b)`: Looks up driving distances between Indian tourist cities.
  2. `order_stops(route)`: Saves the optimized route into session state.
- **Conversational Memory**: Uses `MemorySaver` checkpointer so subsequent user prompts remember previously discussed stops, preferences, and committed routes across multi-turn interactions.
- **Execution Trace**: Visualizes agent reasoning steps, tool calls, tool outputs, and final responses.

In [ ]:
import os
import sys
from dotenv import load_dotenv

# Load environment variables from .env file
load_dotenv()

# Read Groq API configuration
groq_api_key = os.getenv("GROQ_API_KEY")
groq_model = os.getenv("GROQ_MODEL", "qwen/qwen3.6-27b")

print("Environment variables loaded successfully.")
print(f"Groq Model: {groq_model}")


In [ ]:
from langchain_groq import ChatGroq
from tools import get_distance, order_stops, get_saved_routes, clear_saved_routes

# Initialize ChatGroq model
llm = ChatGroq(
    model=groq_model,
    temperature=0,
    api_key=groq_api_key,
)

tools = [get_distance, order_stops]

print("Groq model and tools initialized successfully.")
print(f"Available tools: {[t.name for t in tools]}")


In [ ]:
from langgraph.prebuilt import create_react_agent
from langgraph.checkpoint.memory import MemorySaver

# Initialize MemorySaver for thread-level conversational checkpointing
memory = MemorySaver()

# System prompt guiding agent workflow
system_prompt = (
    "You are an autonomous Multi-Stop Route Planner AI travel agent.\n"
    "Your goal is to plan efficient driving routes for users visiting cities.\n"
    "Workflow Rules:\n"
    "1. When a user provides cities to visit, first check the pairwise distances between them using the `get_distance` tool.\n"
    "2. Evaluate distance data to determine the most efficient sequence of stops minimizing total travel distance.\n"
    "3. Once you determine the optimal order, YOU MUST call the `order_stops` tool to commit/save the route into session state.\n"
    "4. Retain past stops and preferences across conversation turns using memory.\n"
    "5. Present your final answer clearly with the step-by-step route and total distance calculated."
)

# Build ReAct tool-calling agent with checkpointer memory
agent = create_react_agent(
    model=llm,
    tools=tools,
    checkpointer=memory,
    prompt=system_prompt,
)

print("Conversational ReAct Agent initialized with LangGraph MemorySaver checkpointer.")

In [ ]:
from langchain_core.messages import HumanMessage, AIMessage, ToolMessage

def run_agent_turn(agent_app, user_prompt: str, thread_id: str = "session_1"):
    """
    Executes a conversation turn with visible trace (thoughts, tool calls, tool outputs, final answer).
    """
    print("=" * 80)
    print(f"USER PROMPT: {user_prompt}")
    print("=" * 80)
    
    config = {"configurable": {"thread_id": thread_id}}

    # Stream agent events to display execution trace
    events = agent_app.stream({"messages": [HumanMessage(content=user_prompt)]}, config, stream_mode="values")
    
    seen_message_ids = set()
    
    for event in events:
        messages = event.get("messages", [])
        for msg in messages:
            if hasattr(msg, "id") and msg.id in seen_message_ids:
                continue
            if hasattr(msg, "id"):
                seen_message_ids.add(msg.id)
            
            if isinstance(msg, AIMessage):
                if msg.tool_calls:
                    for tc in msg.tool_calls:
                        print(f"\n[AGENT THOUGHT / TOOL CALL]")
                        print(f"  Tool Name : {tc['name']}")
                        print(f"  Arguments : {tc['args']}")
                elif msg.content:
                    print(f"\n[FINAL AGENT RESPONSE]")
                    print(msg.content)
            elif isinstance(msg, ToolMessage):
                print(f"\n[TOOL OUTPUT - {msg.name}]")
                print(f"  Result: {msg.content}")

print("Execution trace helper registered.")

## Turn 1: Planning an Initial Route
The user asks to plan a route starting in **Delhi** and visiting **Agra** and **Jaipur**. 
The agent will look up distances via `get_distance`, find the optimal visiting order, and commit it using `order_stops`.

In [ ]:
# Reset saved routes state before starting demo
clear_saved_routes()

prompt_turn_1 = (
    "Hi! I want to start in Delhi and visit Agra and Jaipur. "
    "Please check driving distances, figure out the most efficient visiting order, "
    "and commit the optimal route."
)

run_agent_turn(agent, prompt_turn_1, thread_id="demo_thread_1")

## Turn 2: Adding a New Stop (Testing Memory Retention)
The user now requests to add **Chandigarh** into the journey.
Because the agent maintains conversational memory via `MemorySaver`, it remembers the previous stops (Delhi, Agra, Jaipur) and computes the updated optimal route before calling `order_stops`.

In [ ]:
prompt_turn_2 = (
    "Actually, I'd also like to add Chandigarh to our trip after starting from Delhi! "
    "Please update our itinerary with Chandigarh included, find the new optimal order, "
    "and save the updated route."
)

run_agent_turn(agent, prompt_turn_2, thread_id="demo_thread_1")

## Turn 3: Querying Session History & Total Distance Calculation
The user asks the agent to review the committed route in memory and summarize the total travel distance.

In [ ]:
prompt_turn_3 = (
    "Can you summarize our final committed route saved in memory and calculate the exact total driving distance?"
)

run_agent_turn(agent, prompt_turn_3, thread_id="demo_thread_1")

In [ ]:
print("\n" + "="*50)
print("SESSION STATE VERIFICATION:")
print("Committed Routes in `SAVED_ROUTES`:")
for idx, r in enumerate(get_saved_routes(), 1):
    print(f"  Route #{idx}: {' -> '.join(r)}")
print("="*50)